# Wav2Lip + GFPGAN Talking Head Server

Runs Wav2Lip (perfect lip sync) + GFPGAN (face enhancement) on a free T4 GPU.

**Faster than LatentSync** (~15-30s per generation).

**Steps:**
1. Run all cells top to bottom
2. Copy the ngrok URL from the last cell output
3. Use that URL in your local `.env` as `COLAB_API_URL`

**Requirements:** Free ngrok account at https://ngrok.com

## 1. Install Dependencies

In [ ]:
# @title Install Wav2Lip + GFPGAN + FastAPI + ngrok
!pip install fastapi uvicorn python-multipart pyngrok aiofiles
!pip install gfpgan basicsr facexlib
!pip install -q face-alignment

import os

# Clone Wav2Lip
if not os.path.exists("Wav2Lip"):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

# Download Wav2Lip checkpoints
os.makedirs("Wav2Lip/checkpoints", exist_ok=True)
if not os.path.exists("Wav2Lip/checkpoints/wav2lip_gan.pth"):
    !wget -q "https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzbfuKlJiR600lQWRxgBbYefRf8sPHLZ2Iz8lbwOQ?e=zRlD0e" -O "Wav2Lip/checkpoints/wav2lip_gan.pth"
if not os.path.exists("Wav2Lip/checkpoints/s3fd-619a316812.pth"):
    !wget -q "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O "Wav2Lip/checkpoints/s3fd-619a316812.pth"

print("Dependencies installed!")

## 2. Start FastAPI Server + ngrok

In [ ]:
# @title Write the API server
%%writefile server.py
import os
import sys
import uuid
import shutil
import tempfile
import subprocess
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
import uvicorn
import cv2
import numpy as np
import face_alignment
import torch

app = FastAPI(title="Wav2Lip + GFPGAN API")

# Face detector
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, device="cuda")


def detect_face_center(image_path: str) -> tuple:
    """Detect face and return bounding box."""
    img = cv2.imread(image_path)
    landmarks = fa.get_landmarks_from_image(img)
    if not landmarks:
        raise ValueError("No face detected in image")
    h, w = img.shape[:2]
    x_min = max(0, int(np.min(landmarks[0][:, 0]) - 50))
    x_max = min(w, int(np.max(landmarks[0][:, 0]) + 50))
    y_min = max(0, int(np.min(landmarks[0][:, 1]) - 50))
    y_max = min(h, int(np.max(landmarks[0][:, 1]) + 50))
    return x_min, y_min, x_max, y_max


def create_video_from_image(image_path: str, audio_path: str, output_path: str):
    """Create a video from a static image with audio duration."""
    import ffmpeg
    probe = ffmpeg.probe(audio_path)
    duration = float(next(s["duration"] for s in probe["streams"] if s["codec_type"] == "audio"))
    (
        ffmpeg
        .input(image_path, loop=1, t=duration)
        .output(audio_path, vcodec="libx264", pix_fmt="yuv420p", r=25)
        .overwrite_output()
        .run(quiet=True)
    )
    # Add audio
    temp_video = output_path + ".temp.mp4"
    os.rename(output_path, temp_video)
    (
        ffmpeg
        .input(temp_video)
        .output(audio_path, vcodec="copy", acodec="copy")
        .overwrite_output()
        .run(quiet=True)
    )
    os.remove(temp_video)


@app.get("/health")
async def health():
    return {"status": "ok", "model": "Wav2Lip + GFPGAN"}


@app.post("/generate")
async def generate(
    image: UploadFile = File(...),
    audio: UploadFile = File(...),
):
    tmp_dir = tempfile.mkdtemp()
    try:
        img_path = os.path.join(tmp_dir, "input.png")
        aud_path = os.path.join(tmp_dir, "input.wav")
        with open(img_path, "wb") as f:
            f.write(await image.read())
        with open(aud_path, "wb") as f:
            f.write(await audio.read())

        # Create video from image + audio
        face_video = os.path.join(tmp_dir, "face_video.mp4")
        create_video_from_image(img_path, aud_path, face_video)

        # Run Wav2Lip
        wav2lip_output = os.path.join(tmp_dir, "wav2lip_output.mp4")
        subprocess.run([
            "python", "Wav2Lip/inference.py",
            "--checkpoint_path", "Wav2Lip/checkpoints/wav2lip_gan.pth",
            "--face", face_video,
            "--audio", aud_path,
            "--outfile", wav2lip_output,
            "--resize_factor", "2",
        ], check=True, capture_output=True)

        # Apply GFPGAN face enhancement
        final_output = os.path.join(tmp_dir, "final_output.mp4")
        subprocess.run([
            "python", "-m", "gfpgan.inference_gfpgan",
            "-i", wav2lip_output,
            "-o", os.path.join(tmp_dir, "enhanced"),
            "-v", "1.3",
            "-s", "2",
            "--bg_upsampler", "realesrgan",
        ], check=True, capture_output=True)

        # Find the enhanced output
        enhanced_dir = os.path.join(tmp_dir, "enhanced")
        for f in os.listdir(enhanced_dir):
            if f.endswith("_out.mp4"):
                shutil.move(os.path.join(enhanced_dir, f), final_output)
                break

        return FileResponse(final_output, media_type="video/mp4", filename="talking_head.mp4")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("Server code written!")

In [ ]:
# @title Start server + ngrok tunnel
import threading
import time
from pyngrok import ngrok

# Set your ngrok authtoken (get it from https://dashboard.ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")  # Uncomment and paste your token

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"YOUR API URL: {public_url}")
print(f"{'='*60}")
print(f"\nCopy this URL and use it in your local .env file:")
print(f"COLAB_API_URL={public_url}")
print(f"\nKeep this cell running! The server is active.")

# Start FastAPI in background thread
def run_server():
    import uvicorn
    uvicorn.run("server:app", host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("\nServer is running!")

## 3. Test the API (Optional)

In [ ]:
# @title Test with sample image + audio
import requests

test_image = "/content/mainimage.png"  # Upload your image first
test_audio = "/content/your-voice.wav"  # Upload your audio first

if os.path.exists(test_image) and os.path.exists(test_audio):
    with open(test_image, "rb") as img, open(test_audio, "rb") as aud:
        response = requests.post(
            f"{public_url}/generate",
            files={"image": img, "audio": aud},
            timeout=300,
        )
    if response.status_code == 200:
        with open("/content/test_output.mp4", "wb") as f:
            f.write(response.content)
        print("Success! Output saved to /content/test_output.mp4")
        print(f"File size: {len(response.content) / 1024 / 1024:.1f} MB")
    else:
        print(f"Error: {response.status_code} - {response.text}")
else:
    print(f"Upload your files first:")
    print(f"  - Image: {test_image}")
    print(f"  - Audio: {test_audio}")
    print("Use the file upload button in the left sidebar.")